# All About Data: Import, Clean, Process

USP 410/510 Urban Data Science — Week 3

Based on McKinney, *Python for Data Analysis*, Chapters 6–8

In [16]:
%pip install pandas geopandas openpyxl 
%pip install fiona


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached fiona-1.10.1.tar.gz (444 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.2-py3-none-any.whl (108 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
  Created wheel for fiona: filename=fiona-1.10.1-cp314-cp314-macosx_26_0_x86_64.whl size=1058796 sha256=855a9ecd842d8d6ef4f9fa86d3a8f8709b2da08894b228987a1f175145a71338
  Stored in directory: /Users/lmwang/Library/Caches/pip/wheels/3c/3e/56/f0e7dc93d0c0f5d2725b85aac3443c34ebc090c73bce308f16
Successfully built fiona
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [fiona]32m4/5 [fiona]

[notice] A new release of pip

## 1. Reading Data into Python

pandas can read from many file formats. The function name always follows the pattern `pd.read_*`.

In [7]:
import pandas as pd

### CSV files

The most common data format. Key parameters:

| Parameter | Purpose |
| :--- | :--- |
| `sep` | Delimiter (default `,`) |
| `encoding` | Character encoding (`utf-8`, `latin1`) |
| `parse_dates` | Columns to convert to datetime |
| `dtype` | Force column types |
| `na_values` | Additional strings to treat as missing |

In [8]:
# Basic read
# df = pd.read_csv('crashes.csv')

# With options — parse dates and preserve leading zeros in ZIP codes
# df = pd.read_csv('crashes.csv',
#                   encoding='utf-8',
#                   parse_dates=['CRASH_DT'],
#                   dtype={'ZIP_CD': str})

### Excel files

In [14]:
# List all sheet names in a workbook
xls = pd.ExcelFile('https://www.oregon.gov/odot/Data/Documents/ODOT_CAR_Unit-Initial_Reported_Motor_Vehicle_Traffic_Fatalities_2025.xlsx')
print(xls.sheet_names)

# Read a specific sheet
# df = pd.read_excel('report.xlsx', sheet_name='Sheet2')

['2025 Weekly Fatals Report']


In [15]:
# Basic read
fatal = pd.read_excel('https://www.oregon.gov/odot/Data/Documents/ODOT_CAR_Unit-Initial_Reported_Motor_Vehicle_Traffic_Fatalities_2025.xlsx')
fatal.head()

,ODOT CAR Unit - Initial Reported Motor Vehicle Traffic Fatalities - Year-to-Date: 2025,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,"Updated April 9, 2026 – Initial information,...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Month,Pedalcyclists,Pedestrians,Motorcyclists,Motor Vehicles,Total Fatalities\n(total of previous columns),CMV Related\n(out of total fatalities),
2,January,1,9,NaN,19,29,5,NaN
3,February,NaN,12,3,20,35,5,NaN
4,March,2,7,3,23,35,4,NaN


**Watch out** with Excel:
- Excel files can have multiple sheets — check which one you need
- Formatting (merged cells, colors) is lost — only values are read
- Dates in Excel can be tricky — always verify with `df.dtypes`

### Geodatabases

ODOT crash data comes as an ESRI **File Geodatabase** (`.gdb`). A geodatabase is like an Excel workbook — one `.gdb` file contains multiple related tables called **layers**.

| Layer | Contains |
| :--- | :--- |
| `CRASH` | One row per crash event |
| `VHCL` | One row per vehicle involved |
| `PARTIC` | One row per participant (driver, pedestrian, etc.) |
| `CRASH_GEOLOC` | Crash locations with geometry |

In [11]:
import geopandas as gpd
import fiona

# List all layers in the geodatabase
fiona.listlayers('data/Statewide_Crashes_2023.gdb')

ModuleNotFoundError: No module named 'fiona'

In [ ]:
# Read the CRASH layer
crashes = gpd.read_file('data/Statewide_Crashes_2023.gdb', layer='CRASH')

### Other data formats

pandas can read from many other sources:

```python
pd.read_json('data.json')                              # JSON
pd.read_parquet('data.parquet')                        # Parquet (fast, compressed)
pd.read_sql('SELECT * FROM crashes', connection)       # SQL databases
pd.read_clipboard()                                    # paste from a spreadsheet
pd.read_html(url)                                      # scrape <table> elements
```

**Tip**: When in doubt, check `pd.read_*` — there are over 15 reader functions.

## 2. Inspecting Your Data

After loading, **always** inspect immediately. These six commands should be your first move with any new dataset.

In [ ]:
crashes.shape              # (rows, columns)

In [ ]:
crashes.columns.tolist()   # column names

In [ ]:
crashes.dtypes             # data types per column

In [ ]:
crashes.head()             # first 5 rows

In [ ]:
crashes.info()             # types, non-null counts, memory usage

In [ ]:
crashes.describe()         # summary statistics for numeric columns

## 3. Data Cleaning

### Missing data

pandas uses `NaN` (Not a Number) to represent missing values.

In [ ]:
# Count missing values per column
crashes.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
# Fraction missing for a specific column
# crashes['SOME_COL'].isna().mean()

In [ ]:
# Strategies for handling missing data:

# Drop rows with any missing values
# df.dropna()

# Drop rows where a specific column is missing
# df.dropna(subset=['CRASH_DT'])

# Fill with a value
# df['SPEED_LMT'].fillna(0)

# Fill with the column median
# df['SPEED_LMT'].fillna(df['SPEED_LMT'].median())

**Key question**: Is the data *missing at random* or is the missingness informative? For example, if crash severity is missing only for minor incidents, dropping those rows biases your analysis toward severe crashes.

### Duplicates

In [ ]:
# Check for fully duplicated rows
crashes.duplicated().sum()

In [ ]:
# Check for duplicated keys
crashes.duplicated(subset=['CRASH_ID']).sum()

In [ ]:
# Remove duplicates (keeps first occurrence)
# df_clean = df.drop_duplicates()

# Remove duplicates on specific columns
# df_clean = df.drop_duplicates(subset=['CRASH_ID'])

Duplicates often appear after merging tables, appending overlapping datasets, or from data entry errors. Always check **after** joins and concatenation.

### Data types and conversion

Getting types right is essential for correct analysis.

| Symptom | Likely cause | Fix |
| :--- | :--- | :--- |
| Numbers with commas | Read as string | `df['col'].str.replace(',', '').astype(float)` |
| Dates as strings | Not parsed on read | `pd.to_datetime(df['col'])` |
| ZIP codes like `7201` | Leading zero dropped | Read with `dtype={'ZIP': str}` |
| Categories as strings | Wastes memory | `.astype('category')` |

In [ ]:
crashes.dtypes

In [ ]:
# Convert types
crashes['CRASH_DT'] = pd.to_datetime(crashes['CRASH_DT'])
# crashes['CRASH_YR'] = crashes['CRASH_YR'].astype(int)
# crashes['SEVERITY'] = crashes['SEVERITY'].astype('category')

### String operations

The `.str` accessor provides vectorized string methods — essential for cleaning text columns.

In [ ]:
# How many unique spellings of city names?
print(f"Unique cities: {crashes['CITY_SECT_NM'].nunique()}")
crashes['CITY_SECT_NM'].value_counts().head(10)

In [ ]:
# Standardize text
crashes['CITY_SECT_NM'] = crashes['CITY_SECT_NM'].str.strip().str.upper()

# Check for patterns
crashes['CITY_SECT_NM'].str.contains('PORTLAND', na=False).sum()

In [ ]:
# Extract parts from datetime
crashes['YEAR'] = crashes['CRASH_DT'].dt.year
crashes['MONTH'] = crashes['CRASH_DT'].dt.month
crashes['HOUR'] = crashes['CRASH_DT'].dt.hour

## 4. Filtering and Selecting

In [ ]:
# Filter rows by condition
portland = crashes[crashes['CITY_SECT_NM'] == 'PORTLAND']
print(f"Portland crashes: {len(portland)} of {len(crashes)} total")

In [ ]:
# Multiple conditions — use & for AND, | for OR
# IMPORTANT: each condition must be wrapped in parentheses
severe_portland = crashes[
    (crashes['CITY_SECT_NM'] == 'PORTLAND') &
    (crashes['CRASH_SVRTY_CD'] == 1)
]
print(f"Severe Portland crashes: {len(severe_portland)}")

In [ ]:
# Select specific columns
subset = crashes[['CRASH_ID', 'CRASH_DT', 'CRASH_SVRTY_CD', 'CITY_SECT_NM']]
subset.head()

In [ ]:
# .query() is often more readable for complex filters
portland = crashes.query("CITY_SECT_NM == 'PORTLAND'")
len(portland)

**Common mistake**: using `and`/`or` instead of `&`/`|`

```python
# Wrong — raises an error
df[df['CITY'] == 'PORTLAND' and df['CRASH_YR'] == 2023]

# Correct
df[(df['CITY'] == 'PORTLAND') & (df['CRASH_YR'] == 2023)]
```

## 5. Combining Data

### Merging (joins)

Combine two DataFrames on a shared key — like SQL joins.

| Type | Keeps |
| :--- | :--- |
| `inner` | Only rows with matching keys in both |
| `left` | All rows from left + matches from right |
| `right` | All rows from right + matches from left |
| `outer` | All rows from both |

In [ ]:
# Read the vehicle layer
vehicles = gpd.read_file('data/Statewide_Crashes_2023.gdb', layer='VHCL')

In [ ]:
# Merge crash data with vehicle data
merged = pd.merge(crashes, vehicles, on='CRASH_ID', how='left')

# Always check: did the number of rows change unexpectedly?
print(f"Crashes: {len(crashes)}, Vehicles: {len(vehicles)}, Merged: {len(merged)}")

### Concatenation

Stack DataFrames vertically (combine years) or horizontally (add columns).

In [ ]:
# Stack rows (e.g., combine multiple years)
# all_years = pd.concat([crashes_2022, crashes_2023, crashes_2024])

# Common pattern: load multiple CSV files
# import glob
# files = glob.glob('data/crashes_*.csv')
# dfs = [pd.read_csv(f) for f in files]
# all_data = pd.concat(dfs, ignore_index=True)

## 6. Grouping and Aggregation

Split-apply-combine: the workhorse of data analysis.

In [ ]:
# Count crashes by severity
crashes.groupby('CRASH_SVRTY_CD').size()

In [ ]:
# Count crashes by month
crashes.groupby('MONTH')['CRASH_ID'].count()

In [ ]:
# Multiple aggregations
crashes.groupby('MONTH').agg(
    total_crashes=('CRASH_ID', 'count'),
    avg_vehicles=('TOT_VHCL_CNT', 'mean')
)

In [ ]:
# Cross-tabulation: crashes by month and severity
pd.crosstab(crashes['MONTH'], crashes['CRASH_SVRTY_CD'])

## 7. Reshaping: Pivot and Melt

Transform between **wide** and **long** formats.

- **Melt** (wide → long): when columns contain values that should be in rows
- **Pivot** (long → wide): when row values should become column headers
- Most plotting libraries prefer **long** format

In [ ]:
# Create a summary table: crashes by city and severity (wide format)
city_severity = crashes.pivot_table(
    index='CITY_SECT_NM',
    columns='CRASH_SVRTY_CD',
    values='CRASH_ID',
    aggfunc='count',
    fill_value=0
)
city_severity.head(10)

In [ ]:
# Melt back to long format
city_severity_long = city_severity.reset_index().melt(
    id_vars=['CITY_SECT_NM'],
    var_name='Severity',
    value_name='Count'
)
city_severity_long.head(10)

## 8. Putting It Together: ODOT Crash Data

A complete workflow from raw data to a simple analysis.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load
crashes = gpd.read_file('data/Statewide_Crashes_2023.gdb', layer='CRASH')
print(f"Loaded {crashes.shape[0]} crashes with {crashes.shape[1]} columns")

In [ ]:
# 2. Inspect
crashes.dtypes

In [ ]:
# 3. Clean
crashes['CRASH_DT'] = pd.to_datetime(crashes['CRASH_DT'])
crashes['CITY_SECT_NM'] = crashes['CITY_SECT_NM'].str.strip().str.upper()

In [ ]:
# 4. Filter to Portland
portland = crashes[crashes['CITY_SECT_NM'] == 'PORTLAND']
print(f"Portland crashes: {len(portland)}")

In [ ]:
# 5. Analyze — crashes by month
monthly = portland.groupby(portland['CRASH_DT'].dt.month)['CRASH_ID'].count()
monthly.plot(kind='bar', title='Portland Crashes by Month (2023)',
             xlabel='Month', ylabel='Number of Crashes')
plt.tight_layout()
plt.show()

## 9. Saving Your Work

Always save intermediate and final results. Keep raw data separate from processed data.

In [ ]:
# To CSV — use index=False unless the index is meaningful
# portland.to_csv('portland_crashes_2023.csv', index=False)

# To Excel
# portland.to_excel('portland_crashes_2023.xlsx', index=False)

# To GeoPackage (for spatial data — replaces Shapefile)
# gdf.to_file('crashes.gpkg', driver='GPKG')

# To Parquet (fast, compressed — good for large datasets)
# portland.to_parquet('portland_crashes_2023.parquet')

## 10. Common Pitfalls

**SettingWithCopyWarning** — pandas' most confusing warning:

```python
# Wrong — may not modify the original DataFrame
df[df['CITY'] == 'PORTLAND']['SEVERITY'] = 'Low'

# Correct — use .loc for assignment
df.loc[df['CITY'] == 'PORTLAND', 'SEVERITY'] = 'Low'
```

**Silent type coercion** — a column with one `NaN` converts integers to floats.

**Memory** — large files may need chunked reading:
```python
for chunk in pd.read_csv('big_file.csv', chunksize=10000):
    process(chunk)
```

## Checklist Before Analysis

- [ ] Checked `df.shape` and `df.dtypes`
- [ ] Inspected missing values with `df.isna().sum()`
- [ ] Verified no unexpected duplicates
- [ ] Confirmed date and numeric types are correct
- [ ] Standardized string columns (case, whitespace)
- [ ] Saved cleaned data separately from raw data